# 06 - LSTM
A configurable same-recording sequence of window features is classified using the last window label. The split is inherited from notebook 00, and preprocessing is fitted on training rows only.

PyTorch is required. Early stopping uses validation F1; the untouched test set is read only after the best checkpoint is restored.

In [ ]:
from pathlib import Path
import sys, copy, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import ConfusionMatrixDisplay
try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
except ImportError as exc:
    raise ImportError('Install requirements.txt to run the LSTM notebook; no neural result was produced.') from exc
ROOT=Path.cwd()
if not (ROOT/'eeg_app').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from eeg_app.src.experiments import load_unbalanced_split, balance_sequences, feature_names, make_sequences, evaluate_predictions, experiment_dirs, seed_everything
seed_everything(42); device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); frame,balance_method=load_unbalanced_split(ROOT); features=feature_names(frame); length=5
train_rows=frame[frame.split.eq('train')]; imputer=SimpleImputer(strategy='median').fit(train_rows[features]); scaler=StandardScaler().fit(imputer.transform(train_rows[features])); scaled=frame.copy(); scaled[features]=scaler.transform(imputer.transform(frame[features])); X,y,meta=make_sequences(scaled,length,features); mask={name:meta.split.eq(name).to_numpy() for name in ('train','validation','test')}; X_train,y_train=balance_sequences(X[mask['train']],y[mask['train']],balance_method,seed=42); X_val,y_val=X[mask['validation']],y[mask['validation']]; X_test,y_test=X[mask['test']],y[mask['test']]
if len(X_train)<20 or len(X_val)<5: print('WARNING: very small train/validation sequence sets; neural metrics are unstable.')
class LSTMClassifier(nn.Module):
    def __init__(self,n_features):
        super().__init__(); self.lstm=nn.LSTM(n_features,32,batch_first=True); self.dropout=nn.Dropout(0.2); self.head=nn.Linear(32,2)
    def forward(self,x): out,_=self.lstm(x); return self.head(self.dropout(out[:,-1]))
model=LSTMClassifier(len(features)).to(device); counts=np.bincount(y_train,minlength=2); weights=torch.tensor([1.0,float(counts[0]/max(1,counts[1]))],dtype=torch.float32,device=device); loss_fn=nn.CrossEntropyLoss(weight=weights); optimizer=torch.optim.Adam(model.parameters(),lr=1e-3,weight_decay=1e-4)
train_loader=DataLoader(TensorDataset(torch.tensor(X_train),torch.tensor(y_train)),batch_size=64,shuffle=True,generator=torch.Generator().manual_seed(42)); val_loader=DataLoader(TensorDataset(torch.tensor(X_val),torch.tensor(y_val)),batch_size=256); history=[]; best_state=None; best_f1=-1; patience=0
for epoch in range(30):
    model.train(); train_loss=0.0
    for xb,yb in train_loader:
        optimizer.zero_grad(); loss=loss_fn(model(xb.to(device)),yb.to(device)); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step(); train_loss+=float(loss.item())*len(yb)
    model.eval(); val_loss=0.0; val_pred=[]
    with torch.no_grad():
        for xb,yb in val_loader:
            logits=model(xb.to(device)); val_loss+=float(loss_fn(logits,yb.to(device)).item())*len(yb); val_pred.extend(logits.argmax(1).cpu().numpy())
    from sklearn.metrics import f1_score
    val_f1=f1_score(y_val,val_pred,zero_division=0) if len(y_val) else 0.0; history.append({'epoch':epoch+1,'train_loss':train_loss/max(1,len(y_train)),'val_loss':val_loss/max(1,len(y_val)),'val_f1':val_f1})
    if val_f1>best_f1: best_f1=val_f1; best_state=copy.deepcopy(model.state_dict()); patience=0
    else: patience+=1
    if patience>=5: break
if best_state is not None: model.load_state_dict(best_state)
model.eval();
with torch.no_grad(): logits=model(torch.tensor(X_test).to(device)); test_pred=logits.argmax(1).cpu().numpy(); test_score=torch.softmax(logits,1)[:,1].cpu().numpy()
metrics=evaluate_predictions(y_test,test_pred,test_score); dirs=experiment_dirs(ROOT); dirs['models'].mkdir(exist_ok=True); dirs['results'].mkdir(exist_ok=True); torch.save({'state_dict':model.state_dict(),'features':features,'sequence_length':length,'imputer':imputer,'scaler':scaler},dirs['models']/'lstm.pt'); out=meta.loc[mask['test'],['patient_id','recording_id','window_id','start_time','end_time','label']].copy(); out['prediction']=test_pred; out['score']=test_score; out.to_csv(dirs['results']/'lstm_test_predictions.csv',index=False); (dirs['results']/'lstm_test_metrics.json').write_text(json.dumps(metrics,indent=2,default=str),encoding='utf-8'); pd.DataFrame(history).to_csv(dirs['results']/'lstm_training_history.csv',index=False); print(json.dumps(metrics,indent=2)); pd.DataFrame(history).plot(x='epoch',y=['train_loss','val_loss']); plt.show(); ConfusionMatrixDisplay.from_predictions(y_test,test_pred,display_labels=['normal','seizure']); plt.show()

In [ ]:
model.eval()
with torch.no_grad():
    validation_logits = model(torch.tensor(X_val).to(device))
    validation_pred = validation_logits.argmax(1).cpu().numpy()
validation_metrics = evaluate_predictions(y_val, validation_pred)
(dirs['results'] / 'lstm_validation.json').write_text(json.dumps({'F1': validation_metrics['F1'], 'Recall': validation_metrics['Recall']}, indent=2), encoding='utf-8')
print('Saved validation selection metrics:', validation_metrics['F1'], validation_metrics['Recall'])